In [2]:
import boto3
import requests
import unicodedata
import csv
import io
from concurrent.futures import ThreadPoolExecutor

In [3]:
BUCKET = "cultivate-mapping-data"
RUN01_PREFIX = "raw/automation/run-01/"
OUTPUT_KEY = "raw/exploration_data/2026_data/04_SHARECITY100/dead_link_report.csv"

s3 = boto3.client("s3", region_name="eu-north-1")

In [4]:
SHARECITY100_PRIORITY4 = [
    "Nairobi", "Dakar", "Johannesburg",
    "Beijing", "Shanghai", "Hong Kong", "Bangalore", "Chennai", "Mumbai",
    "Jakarta", "Tel Aviv", "Tokyo", "Toyama", "Kuala Lumpur", "Manila",
    "Doha", "Singapore", "Seoul", "Dubai",
    "Amsterdam", "Berlin", "Brussels", "Bucharest", "Cologne", "Copenhagen",
    "Frankfurt", "Gothenburg", "Madrid", "Naples", "Nijmegen", "Paris",
    "Prague", "Rome", "Rotterdam", "Stockholm", "Thessaloniki", "Vienna", "Warsaw",
    "Birmingham", "Istanbul", "London", "Moscow", "Zürich",
    "Elora, Ontario", "Montreal", "Toronto", "Vancouver", "Mexico City",
    "Ann Arbor, Michigan", "Asheville, North Carolina", "Atlanta", "Austin, Texas",
    "Berkeley, California", "Bloomington, Indiana", "Boston", "Boulder, Colorado",
    "Chicago", "Cleveland", "Dallas", "Denver", "Detroit", "Gulfport/Biloxi",
    "Hartford, Connecticut", "Houston", "Ithaca, New York", "Jackson, Mississippi",
    "Long Beach, California", "Los Angeles", "Louisville, Kentucky", "Media, Pennsylvania",
    "New York City", "Oakland, California", "Philadelphia", "Pittsburgh",
    "Portland, Oregon", "Rochester, New York", "San Francisco", "Seattle",
    "St. Louis", "Washington, D.C.",
    "Adelaide", "Canberra", "Melbourne", "Sydney", "Christchurch", "Wellington",
    "Buenos Aires", "Santa Cruz de la Sierra", "Porto Alegre", "Rio de Janeiro",
    "São Paulo", "Santiago", "Bogotá", "Medellín", "Quito",
]

In [5]:
def get_run01_keys():
    response = s3.list_objects_v2(Bucket=BUCKET, Prefix=RUN01_PREFIX)
    return [obj["Key"] for obj in response.get("Contents", [])]

In [6]:
run01_keys = get_run01_keys()
print(len(run01_keys))
run01_keys[:5]

208


['raw/automation/run-01/Aalborg_v1.8.0.csv',
 'raw/automation/run-01/Aarhus_v1.8.0.csv',
 'raw/automation/run-01/Adelaide_v1.5.0.csv',
 'raw/automation/run-01/Amsterdam_v1.3.0.csv',
 'raw/automation/run-01/Ankara_v1.4.0.csv']

In [7]:

def normalize(text):
    text = unicodedata.normalize("NFD", text)
    text = "".join(c for c in text if unicodedata.category(c) != "Mn")
    return text.replace("/", "-").lower()

def find_run01_file(city, run01_keys):
    city_norm = normalize(city.split(",")[0])
    for key in run01_keys:
        filename = normalize(key.split("/")[-1].rsplit("_v", 1)[0])
        if city_norm in filename or filename in city_norm:
            return key
    return None

In [8]:
matched = []
unmatched = []
for city in SHARECITY100_PRIORITY4:
    key = find_run01_file(city, run01_keys)
    if key:
        matched.append((city, key))
    else:
        unmatched.append(city)

print(f"Matched: {len(matched)}/95")
print(f"Unmatched: {unmatched}")


Matched: 95/95
Unmatched: []


In [9]:
def extract_urls(key):
    body = s3.get_object(Bucket=BUCKET, Key=key)["Body"].read().decode("utf-8")
    reader = csv.reader(io.StringIO(body))
    header = next(reader)
    url_idx = header.index("URL")
    city_idx = header.index("City")
    name_idx = header.index("Name")
    return [(row[city_idx], row[name_idx], row[url_idx]) 
            for row in reader if row[url_idx].strip()]

In [10]:
city, key = matched[0]
urls = extract_urls(key)
print(f"{city}: {len(urls)} URLs")
urls[:3]

Nairobi: 19 URLs


[('Nairobi', 'Food Banking Kenya', 'https://foodbankingkenya.org'),
 ('Nairobi',
  'Kahurura Community Garden',
  'https://www.fondazioneslowfood.com/en/slow-food-gardens-africa/kahurura-community-garden'),
 ('Nairobi', 'Frī mā mīlā', 'https://www.facebook.com/freemymealKENYA')]

In [11]:
all_urls = []
for city, key in matched:
    all_urls.extend(extract_urls(key))

print(f"Total URLs: {len(all_urls)}")


Total URLs: 5888


In [12]:
def check_url(url, timeout=10):
    try:
        r = requests.head(url, timeout=timeout, allow_redirects=True)
        return r.status_code
    except requests.RequestException:
        return 0

# 먼저 소수로 테스트
test_results = [(url, check_url(url)) for _, _, url in all_urls[:5]]
test_results


[('https://foodbankingkenya.org', 0),
 ('https://www.fondazioneslowfood.com/en/slow-food-gardens-africa/kahurura-community-garden',
  202),
 ('https://www.facebook.com/freemymealKENYA', 200),
 ('https://www.worldfarmersmarketscoalition.org/nairobi-farmers-market-a-fresh-start-for-our-community',
  403),
 ('https://fspnafrica.org/nairobi-amana-csa-day', 200)]

In [13]:
with ThreadPoolExecutor(max_workers=10) as pool:
    statuses = list(pool.map(lambda x: check_url(x[2]), all_urls))

print(f"Done: {len(statuses)} URLs checked")


Done: 5888 URLs checked


In [16]:
output = io.StringIO()
writer = csv.writer(output)
writer.writerow(["city", "name", "url", "status_code", "alive"])

for (city, name, url), status in zip(all_urls, statuses):
    alive = status in range(200, 400) or status in (403, 405, 406)
    writer.writerow([city, name, url, status, alive])

s3.put_object(Bucket=BUCKET, Key=OUTPUT_KEY, Body=output.getvalue().encode("utf-8"))
print(f"Saved to s3://{BUCKET}/{OUTPUT_KEY}")

Saved to s3://cultivate-mapping-data/raw/exploration_data/2026_data/04_SHARECITY100/dead_link_report.csv


In [17]:
from collections import Counter
status_counts = Counter(statuses)
alive_count = sum(1 for s in statuses if s in range(200, 400) or s in (403, 405, 406))
print(f"Alive: {alive_count}/{len(statuses)}")
print(f"Dead: {len(statuses) - alive_count}/{len(statuses)}")
print(f"Status codes: {status_counts.most_common(10)}")

Alive: 5292/5888
Dead: 596/5888
Status codes: [(200, 4383), (403, 616), (404, 303), (0, 212), (406, 170), (202, 94), (405, 29), (503, 14), (500, 11), (429, 11)]
